In [1]:
!pip install transformers datasets evaluate accelerate wandb huggingface_hub -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 2.7 MB/s eta 0:00:00


In [3]:
from kaggle_secrets import UserSecretsClient
import os

secrets = UserSecretsClient()

WANDB_API_KEY = secrets.get_secret("WANDB_API_KEY")
HF_TOKEN = secrets.get_secret("HF_TOKEN")

os.environ["WANDB_API_KEY"] = WANDB_API_KEY
os.environ["HF_TOKEN"] = HF_TOKEN

print("Secrets loaded successfully!")

Secrets loaded successfully!


In [4]:
import torch
from transformers import (
    DistilBertTokenizerFast,
    DistilBertForSequenceClassification
)

In [5]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

cuda


In [7]:
model_name = "distilbert-base-cased"
max_length = 512

In [8]:
id2label = {
    0: "Fantasy",
    1: "Science Fiction",
    2: "Romance",
    3: "Mystery",
    4: "Historical Fiction"
}

label2id = {v:k for k,v in id2label.items()}

In [9]:
tokenizer = DistilBertTokenizerFast.from_pretrained(model_name)

print("Tokenizer loaded successfully!")

tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Tokenizer loaded successfully!


In [10]:
model = DistilBertForSequenceClassification.from_pretrained(
    model_name,
    num_labels=len(id2label),
    id2label=id2label,
    label2id=label2id
).to(device)

print("Model loaded successfully!")

config.json:   0%|          | 0.00/465 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/263M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-cased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
pre_classifier.weight   | MISSING    | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.bias     | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Model loaded successfully!


In [11]:
!pip install datasets evaluate scikit-learn -q

In [12]:
import pandas as pd
import numpy as np
import evaluate
import wandb

from datasets import Dataset
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score
from transformers import TrainingArguments, Trainer

In [15]:
import os

for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

/kaggle/input/datasets/rathodishag25ait2084/dataset/mlops_assignment_2_fine_tuning_classification.py


In [18]:
from datasets import load_dataset

dataset = load_dataset("ag_news")

print(dataset)

README.md: 0.00B [00:00, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/18.6M [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/1.23M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/120000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/7600 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 120000
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 7600
    })
})


In [19]:
import pandas as pd

df = pd.DataFrame(dataset["train"])

df.head()

,text,label
0,Wall St. Bears Claw Back Into the Black (Reute...,2
1,Carlyle Looks Toward Commercial Aerospace (Reu...,2
2,Oil and Economy Cloud Stocks' Outlook (Reuters...,2
3,Iraq Halts Oil Exports from Main Southern Pipe...,2
4,"Oil prices soar to all-time record, posing new...",2


In [20]:
print(df.columns)

Index(['text', 'label'], dtype='object')


In [21]:
from sklearn.model_selection import train_test_split

train_df, test_df = train_test_split(
    df,
    test_size=0.2,
    random_state=42
)

In [22]:
from datasets import Dataset

train_dataset = Dataset.from_pandas(train_df)
test_dataset = Dataset.from_pandas(test_df)

In [23]:
def tokenize_function(example):
    return tokenizer(
        example["text"],
        padding="max_length",
        truncation=True,
        max_length=128
    )

In [24]:
from datasets import load_dataset

dataset = load_dataset("ag_news")

print(dataset)

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 120000
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 7600
    })
})


In [25]:
import pandas as pd

df = pd.DataFrame(dataset["train"])

df.head()

,text,label
0,Wall St. Bears Claw Back Into the Black (Reute...,2
1,Carlyle Looks Toward Commercial Aerospace (Reu...,2
2,Oil and Economy Cloud Stocks' Outlook (Reuters...,2
3,Iraq Halts Oil Exports from Main Southern Pipe...,2
4,"Oil prices soar to all-time record, posing new...",2


In [26]:
print(df.columns)

Index(['text', 'label'], dtype='object')


In [27]:
from sklearn.model_selection import train_test_split

train_df, test_df = train_test_split(
    df,
    test_size=0.2,
    random_state=42
)

print(train_df.shape)
print(test_df.shape)

(96000, 2)
(24000, 2)


In [30]:
train_df = train_df.sample(n=2000, random_state=42)
test_df = test_df.sample(n=500, random_state=42)

In [31]:
from datasets import Dataset

train_dataset = Dataset.from_pandas(train_df)
test_dataset = Dataset.from_pandas(test_df)

In [33]:
def tokenize_function(example):
    return tokenizer(
        example["text"],
        padding="max_length",
        truncation=True,
        max_length=128
    )

In [34]:
train_dataset = train_dataset.map(tokenize_function, batched=True)
test_dataset = test_dataset.map(tokenize_function, batched=True)

Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

Map:   0%|          | 0/500 [00:00<?, ? examples/s]

In [35]:
train_dataset = train_dataset.rename_column("label", "labels")
test_dataset = test_dataset.rename_column("label", "labels")

In [37]:
train_dataset.set_format(
    type="torch",
    columns=["input_ids", "attention_mask", "labels"]
)

test_dataset.set_format(
    type="torch",
    columns=["input_ids", "attention_mask", "labels"]
)

In [38]:
import wandb

wandb.login()

/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from WANDB_API_KEY.
wandb: Currently logged in as: g25ait2084 (g25ait2084-iit) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

In [39]:
wandb.init(
    project="mlops-assignment2",
    name="distilbert-run-1",
    config={
        "model": "distilbert-base-cased",
        "epochs": 3,
        "batch_size": 8,
        "learning_rate": 3e-5,
        "max_length": 128,
        "platform": "Kaggle"
    }
)

In [40]:
from sklearn.metrics import accuracy_score, f1_score

def compute_metrics(pred):
    labels = pred.label_ids
    preds = pred.predictions.argmax(-1)

    return {
        "accuracy": accuracy_score(labels, preds),
        "f1": f1_score(labels, preds, average="weighted")
    }

In [42]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./results",
    num_train_epochs=3,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    warmup_steps=100,
    weight_decay=0.01,
    logging_steps=50,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    report_to="wandb",
    run_name="distilbert-run-1"
)

In [43]:
from transformers import Trainer

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    compute_metrics=compute_metrics,
)

In [44]:
trainer.train()

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Epoch,Training Loss,Validation Loss,Accuracy,F1
1,1.066611,1.178533,0.816000,0.816467
2,0.637603,0.870137,0.860000,0.859118
3,0.242825,1.088526,0.864000,0.862643


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.weight', 'distilbert.embeddings.LayerNorm.bias'].
There were unexpected keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.beta', 'distilbert.embeddings.LayerNorm.gamma'].


TrainOutput(global_step=375, training_loss=0.891707176208496, metrics={'train_runtime': 70.5572, 'train_samples_per_second': 85.037, 'train_steps_per_second': 5.315, 'total_flos': 198711728640000.0, 'train_loss': 0.891707176208496, 'epoch': 3.0})

In [45]:
eval_results = trainer.evaluate()

print(eval_results)

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


{'eval_loss': 0.8700025081634521, 'eval_accuracy': 0.86, 'eval_f1': 0.8591175582852096, 'eval_runtime': 1.7287, 'eval_samples_per_second': 289.234, 'eval_steps_per_second': 18.511, 'epoch': 3.0}


In [47]:
wandb.log({
    "final/loss": eval_results["eval_loss"],
    "final/accuracy": eval_results["eval_accuracy"],
    "final/f1": eval_results["eval_f1"],
})

In [48]:
from sklearn.metrics import classification_report
import json
import numpy as np

predictions = trainer.predict(test_dataset)

preds = np.argmax(predictions.predictions, axis=-1)

labels = predictions.label_ids

report = classification_report(labels, preds, output_dict=True)

print(report)

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


{'0': {'precision': 0.8770491803278688, 'recall': 0.7985074626865671, 'f1-score': 0.8359375, 'support': 134.0}, '1': {'precision': 0.9274193548387096, 'recall': 0.9504132231404959, 'f1-score': 0.9387755102040817, 'support': 121.0}, '2': {'precision': 0.8363636363636363, 'recall': 0.7796610169491526, 'f1-score': 0.8070175438596491, 'support': 118.0}, '3': {'precision': 0.8055555555555556, 'recall': 0.9133858267716536, 'f1-score': 0.8560885608856088, 'support': 127.0}, 'accuracy': 0.86, 'macro avg': {'precision': 0.8615969317714427, 'recall': 0.8604918823869674, 'f1-score': 0.8594547787373349, 'support': 500.0}, 'weighted avg': {'precision': 0.8614775934917659, 'recall': 0.86, 'f1-score': 0.8591175582852096, 'support': 500.0}}


In [49]:
with open("eval_report.json", "w") as f:
    json.dump(report, f, indent=2)

print("Evaluation report saved!")

Evaluation report saved!


In [50]:
artifact = wandb.Artifact(
    "eval-report",
    type="evaluation"
)

artifact.add_file("eval_report.json")

wandb.log_artifact(artifact)

print("Artifact uploaded!")

Artifact uploaded!


In [51]:
wandb.finish()

eval/accuracy,▁▇█▇
eval/f1,▁▇█▇
eval/loss,█▁▆▁
eval/runtime,▇▁▂█
eval/samples_per_second,▁█▇▁
eval/steps_per_second,▁█▇▁
final/accuracy,▁▁
final/f1,▁▁
final/loss,▁▁
test/accuracy,▁
+10,...


In [52]:
from huggingface_hub import login

login(token=HF_TOKEN)

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


In [55]:
model.push_to_hub(
    "IshaIIT/distilbert-news-classifier"
)

tokenizer.push_to_hub(
    "IshaIIT/distilbert-news-classifier"
)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

README.md: 0.00B [00:00, ?B/s]

CommitInfo(commit_url='https://huggingface.co/IshaIIT/distilbert-news-classifier/commit/a4dac5808b7b16f7fa7c62a13a69d5ffe1367a9f', commit_message='Upload tokenizer', commit_description='', oid='a4dac5808b7b16f7fa7c62a13a69d5ffe1367a9f', pr_url=None, repo_url=RepoUrl('https://huggingface.co/IshaIIT/distilbert-news-classifier', endpoint='https://huggingface.co', repo_type='model', repo_id='IshaIIT/distilbert-news-classifier'), pr_revision=None, pr_num=None)